# Getting started with swapnet

**One QAOA cost layer, laid out and routed onto a real device within a budget of `k` parallel SWAP layers.**

A dense problem does not fit. 100 variables means 4,950 two-body terms and a Heron device has ~175 couplings, so the question is never "can we route it" but **which terms are worth the depth**. swapnet answers that with a quadratic assignment for the layout and a prize-collecting vertex-disjoint multi-commodity flow for the routing.

It is designed to be **one stage of a larger pipeline** — alongside parameter-setting and LCU modules — so the artifacts between stages matter more than the API. They are declared, validated and JSON-serialisable, and the routing artifact is deliberately algorithm-neutral so swapnet can be replaced without touching anything downstream.

## The pipeline

Six stages, each a separate call. There is deliberately **no single entry point** that does all of it: the interesting decisions live between the stages.

| # | stage | call | artifact out |
| --- | --- | --- | --- |
| 0 | problem | `Problem.from_covariance` / `.from_qubo` / `.from_matrix` | `Problem` |
| 1 | shift | `shifts.rank0` / `shifts.rank1` | `Shift` + residual `Problem` |
| 2 | hardware | `Hardware.from_backend` / `.from_target` / `.from_error_map` | `Hardware` |
| 3 | layout | `map_layout` | `MappingResult` |
| 4 | **routing** | `route` → `schedule.from_routing_plan`, or `plan_schedule` | **`Schedule`** |
| 5 | ansatz | `ansatz.build` | `Ansatz` |

Every artifact has `to_dict()` / `from_dict()`, so any stage boundary can be a JSON file between two separately-written modules. `Ansatz` holds a Qiskit circuit, so it pickles rather than serialising to JSON.

**`Schedule` is the pivot.** Stage 5 consumes only a `Schedule`, so a different circuit-optimisation algorithm plugs in by emitting one.

The core of swapnet is Qiskit-free. Stages 2 and 5 import Qiskit lazily, inside the functions that need it, so Qiskit is only required if you want a `Hardware` from a backend or an actual circuit. This guide uses both, and runs offline with no IBM Quantum account, using a fake backend. The whole sequence takes about a minute, nearly all of it in the two layout searches.

## Stage 0 — the problem

`Problem` holds the Ising instance, with couplings `J`, local fields `h` and a constant `offset`. Here we build one from a synthetic equity-like covariance matrix: a positive market factor, sector factors and dispersed volatilities.

In [1]:
import numpy as np
import swapnet


def factor_covariance(n=100, seed=7, sectors=4, vol_sigma=0.8):
    """Equity-like: a positive market factor, sector factors, dispersed vols."""
    rng = np.random.default_rng(seed)
    columns = [rng.uniform(0.55, 0.95, size=n)]  # the market
    labels = rng.integers(0, sectors, size=n)
    for sector in range(sectors):
        columns.append(np.where(labels == sector, rng.uniform(0.15, 0.35, size=n), 0.0))
    loadings = np.column_stack(columns)
    corr = loadings @ loadings.T
    np.fill_diagonal(corr, np.diag(corr) + rng.uniform(0.1, 0.4, size=n))
    scale = np.sqrt(np.diag(corr))
    corr /= np.outer(scale, scale)
    volatility = np.exp(rng.normal(0.0, vol_sigma, size=n))  # the part that matters
    return corr * np.outer(volatility, volatility)


sigma = factor_covariance()
problem = swapnet.Problem.from_covariance(sigma, sigma_scale=0.5, label="factor100")

In [2]:
print(problem)
print(problem.verify(sigma))  # ~1e-16: H(z) == f(x), so h and offset are right
print(problem.concentration())  # pairs holding 50/90/99% of sum J^2

<Problem factor100 n=100 pairs=4950 (complete) sumJ2=1623.49>
3.636987417770825e-16
{'num_pairs': 4950, 'total_j_squared': 1623.4917890544186, 'abs_j_min': 0.005078012082991184, 'abs_j_median': 0.15903109860346876, 'abs_j_max': 6.854928034502738, 'num_negative_couplings': 0, 'pairs_for_50pct_of_j2': 84, 'pairs_for_90pct_of_j2': 828, 'pairs_for_99pct_of_j2': 2583}


`verify` is worth running once per instance. A wrong `h` or `offset` leaves `J` correct, so routing, depth and coverage all look fine and only the energies are wrong — silently.

> **`Problem` owns the coupling-power asymmetry, and this is the one trap in the library.** The mapper's objective is *linear* in the couplings it is given; the router's reward and its `realized_weight` are *quadratic*. Both target `sum J^2`, so the mapper needs `J**2` and the router needs `|J|`. Use `problem.mapper_couplings()` and `problem.router_couplings()` and you cannot get it wrong. Hand `J**2` to the router yourself and it silently optimises `sum J^4` and reports coverage of *that* — the plan still validates, the gate counts still look reasonable, and the objective is not the one you are scoring.

## Stage 1 — the shift

This stage is optional, and it dominates everything else. Two parts of `J` can be implemented **analytically, in one layer of RZ rotations**, with no two-qubit gates and no routing:

```text
rank 0    J_ij ~ c           sum_{i<j} z_i z_j          = ((sum_i z_i)^2 - n) / 2
rank 1    J_ij ~ b_i b_j     sum_{i<j} b_i b_j z_i z_j  = ((b . z)^2 - |b|^2) / 2
```

Both are functions of a single weighted sum of `z`, which is why they cost one rotation layer rather than a network. Fitting them means minimising what is left, in the same measure the depth budget is scored in:

```text
rank 0    min_c  sum_{i<j} (J_ij - c)^2        closed form: c* = mean J_ij
rank 1    min_b  sum_{i<j} (J_ij - b_i b_j)^2  alternating fit, diagonal free
```

In [3]:
s0 = swapnet.shifts.rank0(problem)
s1 = swapnet.shifts.rank1(problem)

print(s0)  # absorbs ~29.5% of sum J^2
print(s1)  # absorbs ~99.8% of sum J^2
print(s1.operator)  # the identity to implement
print(s1.metadata["plain_top_eigenpair_fraction"])  # 0.9913 -- see below

residual = s1.residual(problem)  # a Problem again: hand this to routing

<Shift rank0 n=100 absorbs=29.50% of sum J^2>
<Shift rank1 n=100 absorbs=99.76% of sum J^2>
sum_{i<j} b_i b_j z_i z_j = ((b . z)^2 - |b|^2) / 2
0.9912629435060198


**Rank 0 is exact accounting, not an approximation.** `c* = mean J` makes the split orthogonal, because `sum d = 0` kills the cross term:

```text
sum J^2 = m c*^2 + sum d^2
```

so `absorbed_fraction` is `m c*^2 / sum J^2` with no error term. `shifts.rank0` asserts this.

**Rank 1 fits with a free diagonal, and that is the whole trick.** The objective runs over `i < j` only, so `b_i^2` is unconstrained and must not be fitted to anything. A single eigendecomposition implicitly fits a zero diagonal it is never scored on; alternating between the top eigenpair and resetting `diag <- b_i^2` does not. Measured on this problem: **99.76% vs 99.13%** for the plain top eigenpair, in 28 alternations.

Both refuse a sparse problem rather than densifying it — the operators run over *every* pair, so shifting a sparse `J` would invent couplings that were not in it.

## Stage 2 — the hardware

`Hardware` turns a backend into a live coupling graph and per-link traversal costs `W_ij`. We use `FakeBoston`, a 156-qubit fake backend, so no account is needed.

In [4]:
from qiskit_ibm_runtime.fake_provider import FakeBoston

hardware = swapnet.Hardware.from_backend(FakeBoston())
print(hardware)  # qubits, live links, dead links, usable qubits
print(hardware.dead_links)  # dead for cz
print(hardware.unusable_qubits)  # qubits isolated by dead links
w = hardware.edge_weights  # W_ij: {(i, j): cost}, pass to map_layout and route
print(len(w), min(w.values()), max(w.values()))  # mean is 1.0

<Hardware fake_boston qubits=156 live_links=175 dead=1 usable=156>
((84, 85),)
[]
175 0.12692057415179453 30.31357280391441


Three things this does that are easy to get wrong by hand:

- **Dead links are per gate, not per device.** A link qualifies only when *every* two-qubit gate the circuit needs is live on it, and its cost is the worst of them. Pricing one gate lets a cost term land on a link that cannot run it. Pass `gates=[...]` to be explicit.
- **The planning graph is the largest connected component.** Dead links can isolate a qubit (`FakeTorino` loses 4 that way). Routing needs one component; the rest are reported as `unusable_qubits`, never silently dropped.
- **`W_ij = -log(1 - error)`, normalised to mean 1.** That transform is additive along a path exactly when fidelity is multiplicative, so minimising summed cost maximises the fidelity of the route. Mean-1 keeps every ratio and only changes the unit, which matters because the search schedule is calibrated against unit-cost hops.

For a frozen calibration snapshot or a hand-built topology, use `Hardware.from_error_map({(i, j): error}, num_qubits=...)`. The backend object is not stored on the artifact — a `Target` is large and version-coupled, and keeping one would pin every consumer to this Qiskit build.

## Stages 3 and 4 — layout and routing

Run them separately when you want to inspect or replace a stage. Note which couplings each stage receives: `mapper_couplings()` (`J**2`, for the linear objective) for the layout, and `router_couplings()` (tilted, for the quadratic reward) for the routing.

In [5]:
interaction = problem.interaction_graph()

mapping = swapnet.map_layout(  # stage 3
    hardware.graph,
    interaction,
    seed=12345,
    max_seconds=20.0,
    edge_weights=hardware.edge_weights,
    couplings=problem.mapper_couplings(),  # J**2   -- linear objective
)

plan = swapnet.route(  # stage 4
    hardware.graph,
    interaction,
    mapping,
    seed=12345,
    max_depth=4,
    edge_weights=hardware.edge_weights,
    couplings=problem.router_couplings(),  # tilted -- quadratic reward
    options=swapnet.RouteOptions(
        max_commodities_per_round=120,
        routing=swapnet.RoutingOptions(max_detour=2),
    ),
)
swapnet.verify_plan(hardware.graph, interaction, plan, require_converged=False)
schedule = swapnet.schedule.from_routing_plan(plan, problem.n)

Or take the default wiring, which is the same two calls plus the validation. `map_seconds` defaults to 60; it is pinned to 20 here so that every number quoted below is reproducible, and because the assignment cost saturates within seconds anyway.

From here on we route the **rank-1 residual**, which is the part of the problem that nothing else can implement.

In [6]:
schedule = swapnet.plan_schedule(residual, hardware, k=4, seed=12345, map_seconds=20.0)
report = swapnet.schedule.validate(schedule, residual.n, hardware, k=4)
print(report["num_rounds"], "rounds,", report["num_swap_layers"], "swap layers,", report["num_swaps"], "swaps,", report["num_executed_pairs"], "pairs executed")

5 rounds, 4 swap layers, 38 swaps, 180 pairs executed


**Sweeping `k` costs one routing run, not one per `k`.** swapnet's loop is greedy per round and `max_depth` only stops it, so the plan for budget `k` is the `k`-layer prefix of the plan for any larger budget:

In [7]:
plans = swapnet.plan_schedule_sweep(
    residual, hardware, [0, 1, 2, 4, 8], seed=12345, map_seconds=20.0
)

`tests/test_pipeline.py` checks this by planning at each `k` separately and comparing, so it is a verified property and not an assumption. An algorithm that reorders its rounds under a smaller budget does not have it and must be re-invoked per `k`.

## Stage 5 — the ansatz

`ansatz.build` turns a `Schedule` into the circuit. `angles_from_problem` converts the problem's coefficients into gate angles for a given `gamma`, and `verify_structure` checks the built circuit.

In [8]:
layer = swapnet.ansatz.build(schedule, residual, hardware, k=4)
one_body, two_body = swapnet.ansatz.angles_from_problem(layer, residual, gamma=0.1)
swapnet.ansatz.verify_structure(layer, one_body, two_body)
circuit = layer.bind(one_body, two_body)

The `Ansatz` carries everything a downstream module needs:

- `layer.circuit` — `U(theta) = exp(-i G(theta)/2)`, one layer, on device qubits, with the SWAP network. No state, no mixer, no measurement.
- `layer.generator` — `G(theta)` on `n` qubits; the coefficients *are* the circuit's `Parameter`s.
- `layer.couplings` — the two-body terms that survived the budget.
- `layer.input_layout` — `pi_0`: problem qubit → device qubit.
- `layer.output_layout` — `pi_fin`: `pi_0` composed with this layer's SWAP network.
- `layer.mask` — `(n, n)` symmetric bool: which terms survived.
- `layer.metadata` — device, budget, `covered_j2_fraction`, producer, convention.

In [9]:
print(layer.metadata)
print("terms kept:", len(layer.couplings), "of", problem.n * (problem.n - 1) // 2)
print("parameters:", layer.circuit.num_parameters)
print(dict(circuit.count_ops()))

{'problem_label': 'factor100-rank1residual', 'num_problem_qubits': 100, 'device': 'fake_boston', 'device_qubits': 156, 'k_swap_layers_allowed': 4, 'swap_layers_used': 4, 'num_swaps': 38, 'num_terms_kept': 180, 'num_terms_total': 4950, 'covered_j2': 1.9332431615767607, 'total_j2': 3.922725019430539, 'covered_j2_fraction': 0.49283168001855227, 'convention': 'U(theta) = exp(-i G/2), G = sum_alpha theta_alpha Z^alpha', 'producer': 'swapnet.plan_schedule'}
terms kept: 180 of 4950
parameters: 280
{'rzz': 180, 'rz': 100, 'swap': 38}


At `k=4` on the rank-1 residual, this keeps 180 terms of 4,950 using 38 swaps in 4 layers and 280 parameters, with `{'rzz': 180, 'rz': 100, 'swap': 38}` once bound. These numbers move with `map_seconds` — at the 60 s default the same call finds a different layout and spends 40 swaps for the same 180 terms. The layout search is stochastic in its time budget, so pin `map_seconds` alongside `seed` when you need bit-identical plans.

**The two-body terms in a round are emitted grouped by an edge colouring, which minimises the round's depth.** Their order is free to choose — they are diagonal, so they commute exactly and no ordering changes the unitary — but it decides the depth, and nothing downstream recovers it: Qiskit's depth is the DAG longest path, and neither the transpiler nor a scheduling pass reorders commuting gates.

A layer of two-qubit gates is a matching, so a depth-`d` schedule of a round's terms *is* a proper `d`-edge-colouring, and the minimum achievable depth is exactly the edge chromatic number of that round's interaction subgraph. `ansatz.edge_coloring` attains it on bipartite graphs via `rustworkx.graph_bipartite_edge_color`, where König makes the chromatic number equal the maximum degree; every IBM heavy-hex map is bipartite and subgraphs inherit that, so a round on hardware is exactly optimal. Otherwise Misra–Gries bounds it at `Delta + 1`.

Each round is coloured on its own terms. Both halves matter: a SWAP does not commute with an `rzz`, so no term may cross a swap layer, and colouring the device instead would size every round's palette by the device's degree rather than the round's own. At `k=0` on a degree-3 device that is 3 `rzz` layers, so **6** two-qubit layers once each `rzz` becomes two `cz`.

`verify_structure` walks the real 100-qubit circuit and checks that every term is on the right qubits exactly once and that the SWAP network lands on the advertised `output_layout`. It **cannot** catch a wrong angle convention — it computes what it expects with the same helpers the builder used, so the two move together. Only `tests/test_ansatz.py` can, by comparing against `expm(-i G/2)` computed independently.

## The angle convention

```text
U(theta) = exp(-i G(theta) / 2)          G(theta) = sum_alpha theta_alpha Z^alpha
```

`alpha` runs over the Z terms the circuit realises: `Z_i` and `Z_i Z_j`. The `/2` is part of the definition of the generator — reading `U = exp(-i G)` is wrong by a factor 2.

**There is no conversion factor to lose.** Qiskit's rotations are defined with the same sign and half-angle, `RZ(t) = exp(-i t Z/2)` and `RZZ(t) = exp(-i t Z Z/2)`, so the gate angle *is* `theta`. `swapnet.rz_angle` and `swapnet.rzz_angle` are the identity. They exist anyway so there is one authority to cite and one file to change if the convention ever moves.

For QAOA, one cost layer of `H_C = sum J_ij Z_i Z_j + sum h_i Z_i` at angle `gamma` is `exp(-i gamma H_C)`. Matching that to `exp(-i G/2)` gives `G = 2 gamma H_C`, so

```text
theta_alpha = 2 * gamma * coeff_alpha
```

`ansatz.angles_from_problem` applies exactly this, so a parameter-setting module supplies `gamma` and never handles the factor itself. `swapnet.gamma_to_theta` is the same relation for one coefficient.

## Conclusion

The sections below step back from the mechanics and look at what the pipeline buys you, which knobs steer it, and where it can be replaced or improved.

### What the numbers actually say

The results below are for the synthetic 100-variable problem used throughout this guide, on `FakeBoston`, seed 12345, `map_seconds=20`. Each variant is one `plan_schedule_sweep` call, about 23 s. `routed` is the share of the *residual* the circuit realises; `TOTAL` is the share of the original `sum J^2` once the analytically-implemented part is counted.

| variant | k | layers | swaps | terms | routed | **TOTAL** |
| --- | --- | --- | --- | --- | --- | --- |
| raw `J` | 0 | 0 | 0 | 108 | 13.19% | **13.19%** |
| raw `J` | 2 | 2 | 9 | 132 | 24.79% | **24.79%** |
| raw `J` | 8 | 8 | 57 | 210 | 38.59% | **38.59%** |
| rank-0 residual | 0 | 0 | 0 | 109 | 16.37% | **41.04%** |
| rank-0 residual | 2 | 2 | 10 | 133 | 27.73% | **49.05%** |
| rank-0 residual | 8 | 8 | 62 | 210 | 41.36% | **58.66%** |
| rank-1 residual | 0 | 0 | 0 | 106 | 27.30% | **99.82%** |
| rank-1 residual | 2 | 2 | 16 | 144 | 42.25% | **99.86%** |
| rank-1 residual | 8 | 8 | 89 | 245 | 57.88% | **99.90%** |

Read frankly, this says something uncomfortable about swapnet's own value on this objective:

- **The shift dominates the routing, by a lot.** A rank-1 shift at `k=0` — no swaps at all — reaches 99.82%, against 38.59% for eight SWAP layers of routing on raw `J`. If you are scoring `sum J^2` and you can implement the rank-1 term, the depth budget stops being the interesting variable: `k=0` to `k=8` buys 0.08 percentage points.
- **Routing still earns its keep on the residual**, which is where it should be judged: 27.30% → 57.88% as `k` goes 0 → 8. That is a real factor of 2, on the part nothing else can implement.
- **Rank 0 is worth having and is nearly free**, but it is not close to rank 1 here. It absorbs 29.50% because a single constant cannot follow the volatility scaling; `b_i b_j` can, since `cov_ij ~ vol_i vol_j corr_ij`. On a problem whose volatilities are uniform, expect rank 0 to do much better and the gap to close.

**And the caveat that bounds all of it.** `sum J^2` is a Frobenius-type measure on the couplings. Absorbing 99.76% of it does **not** mean the shifted term reproduces the energy landscape to 99.76%, and it does not mean the residual is negligible to the optimiser — the residual still contains every pair, just with small individual weights. It also assumes the analytic module actually implements the rank-1 term, which has its own cost that swapnet knows nothing about. What the number does measure is exactly what a depth budget is scored on, which is why it is the one reported.

For calibration: on the real F2 100-asset covariance the same fits give **29.68% rank 0 and 99.75% rank 1** (plain top eigenpair 96.63%), so the synthetic problem sits in the same regime rather than flattering the method.

### Tuning knobs

The knobs below are reachable through `plan_schedule` by name, or on `RouteOptions`, `RoutingOptions` and `MapParams` when running the stages by hand. **The defaults listed are `plan_schedule`'s**, which are not always the native ones: `plan_schedule` sets `max_detour=2` and `max_commodities_per_round=120`, where bare `RouteParams` leaves both uncapped (`-1` and `0`). Running stages 3 and 4 by hand means setting them yourself.

| knob | default | what it does |
| --- | --- | --- |
| `k` / `max_depth` | `None` (unlimited) | **The budget**, in parallel SWAP layers. `0` forbids every swap; a round may still execute its pairs, because terms cost no budget. |
| `max_detour` | `2` | Cap on how far past the shortest path an arc may go. **Not a preference.** Without it the per-round MILP on a dense problem does not solve inside the time limit, and a round that times out routes *nothing* — measured 0.1 s with the cap vs a 60 s timeout without. |
| `tilt` | `4.0` | Reward compression. `router_couplings` returns `sqrt(1 + tilt*J^2/max)`, so reward is `1 + tilt*J^2/max`: every pair can pay its way while the heaviest keep `1+tilt` times the priority. Without it, routing starves: `J^2` spans 6.3 orders of magnitude on this problem, and after mean-1 normalisation **36% of pairs carry less reward than a single arc costs**, so the router declines them and a budget increase buys nothing (on the real F2 covariance: 5.7 orders, 26%). `tilt=None` gives untilted `|J|`, which makes `realized_weight` *be* the `sum J^2` coverage — the honest choice when you want the reported number to be the objective rather than a monotone proxy for it. |
| `max_commodities_per_round` | `120` | How many pending pairs are offered to each round's MILP. Terminal capacity allows one routed commodity per qubit per round, so raising this costs solve time without buying reservations. |
| `max_swaps_per_commodity` | `2` (upstream) | Hops per reserved path per round; `4` is upstream's `run(4)`. Measured: parallelism per layer *falls* (a pair stops as soon as its endpoints are adjacent, so the second layer is half empty) and depth granularity coarsens into a step function. It does buy ~22% fewer two-qubit gates and ~35% faster planning. Set it when gate count binds, not when depth does. |
| `route_reward` | `30.0` | Base reward per routed commodity, before the `J^2` scaling. Traded off against arc costs, so it sets how long a path the router will pay for. |
| `map_seconds`, `map_params` | `60.0`, upstream | Layout search budget. The assignment cost saturates within seconds and coverage is not monotone in the search budget, so more is usually not better. |
| `seed` | `0` | Reproducibility, for both the annealing schedule and MILP tie-breaking. |

**The layout carries most of the coverage, and its objective is only a proxy.** The mapper minimises `sum J_ij^2 * distance(i,j)` while what gets scored is `sum J^2` over pairs at distance exactly 1. On a *complete* interaction graph the unweighted assignment objective is exactly invariant under permuting variables within a fixed qubit set — it carries no signal at all — which is why weighting it by `J^2` is not optional.

### Replacing swapnet

Stage 5 consumes only a `Schedule`, so a different circuit-optimisation algorithm plugs into the rest of the pipeline by emitting one. The contract is a function with this shape:

```python
def plan(problem: Problem, hardware: Hardware, k: int, seed: int) -> Schedule: ...
```

A `Schedule` is an initial layout plus ordered rounds of *"realise these variable pairs, then apply this qubit-disjoint layer of swaps"*:

```python
from swapnet.schedule import Round, Schedule, validate

schedule = Schedule(
    initial_layout={variable: qubit, ...},   # injective; every variable placed; idle
                                             # qubits may be included to swap through
    rounds=[Round(pairs=[(0, 1), (2, 3)], swaps=[(0, 1), (2, 3)]),
            Round(pairs=[(0, 3)])],
    info={"producer": "my algorithm", "seed": 7},
)
validate(schedule, problem.n, hardware, k=2)
```

The snippet above is a template and is not meant to be run as written: `variable` and `qubit` are placeholders for your own layout.

`validate` enforces five rules, and rejects rather than repairs:

1. `initial_layout` is injective into `0..num_qubits-1` and places every variable.
2. Every executed pair is on a live link **at the moment it executes**.
3. Every swap is on a live link, and one round's swaps are qubit-disjoint.
4. No pair is executed twice — a cost layer applies each term once.
5. At most `k` rounds carry swaps.

Run it on any schedule you did not produce yourself, and on your own before building a circuit. It is the only thing between a subtly wrong routing algorithm and a circuit that runs while computing the wrong generator.

swapnet's own `RoutingPlan` is richer — per-round MILP statistics, reservation counts, realized weight — and is deliberately *not* part of the contract: it is a diagnostic of one method. `schedule.from_routing_plan` is the adapter down to the neutral form. The test suite includes a hand-written odd-even transposition network that passes the same validator, as a worked non-swapnet producer.

### Where routing is weak, if you want to beat it

Replacing swapnet is most useful where its routing is weakest, so it is worth knowing where that is. Measured: swapnet gains 6 to 20 new terms per SWAP layer, median 12. A complete linear swap network on 100 qubits brings `n/2 = 50` new pairs together per layer.

swapnet uses far fewer swaps per layer because it reserves a **full vertex-disjoint path** for each routed pair and then advances it by one hop — so a length-10 path blocks 10 qubits to gain one adjacency, and only a handful of pairs fit per round. `max_swaps_per_commodity` advances each reservation further; it does not create more of them. Relaxing the reservation is the thing that would.

So an alternative router is useful if it can bring more pairs together per SWAP layer than path reservation allows. Emit a `Schedule`, run it through `validate`, and the rest of the pipeline, including the ansatz builder, works unchanged.